![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [108]:
df = pd.read_csv("bank_marketing.csv")
for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
mortgage
--------------
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64
previous_outcome
--------------
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
campaign_outcome
--------------
no     36548
yes     4640
Name: campaign_outcome, dtype: int64


In [109]:
import pandas as pd
import numpy as np

# Start coding here...
df = pd.read_csv("bank_marketing.csv")

df.head(30)

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no
5,5,45,services,married,basic.9y,unknown,no,may,5,198,1,0,nonexistent,93.994,4.857,no
6,6,59,admin.,married,professional.course,no,no,may,3,139,1,0,nonexistent,93.994,4.857,no
7,7,41,blue-collar,married,unknown,unknown,no,may,12,217,1,0,nonexistent,93.994,4.857,no
8,8,24,technician,single,professional.course,no,yes,may,21,380,1,0,nonexistent,93.994,4.857,no
9,9,25,services,single,high.school,no,yes,may,5,50,1,0,nonexistent,93.994,4.857,no


In [110]:
#client
client = df[['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage']]

#cleaning part
#job column
client['job'] = client['job'].str.replace('.', '_')

#education column
client['education'] = client['education'].str.replace('.', '_')
client['education'] = client['education'].replace('unknown', np.nan)  

#credit_default column
client['credit_default'] = client['credit_default'].map({'yes': 1, 'no': 0, 'unknown': 0}).astype('bool')

#mortgage column
client['mortgage'] = client['mortgage'].map({'yes': 1, 'no': 0, 'unknown': 0}).astype('bool')

#Checking
print(client['credit_default'].unique())
print(client.head(10))
print(client.dtypes)

[False  True]
   client_id  age          job  ...            education credit_default  mortgage
0          0   56    housemaid  ...             basic_4y          False     False
1          1   57     services  ...          high_school          False     False
2          2   37     services  ...          high_school          False      True
3          3   40       admin_  ...             basic_6y          False     False
4          4   56     services  ...          high_school          False     False
5          5   45     services  ...             basic_9y          False     False
6          6   59       admin_  ...  professional_course          False     False
7          7   41  blue-collar  ...                  NaN          False     False
8          8   24   technician  ...  professional_course          False      True
9          9   25     services  ...          high_school          False      True

[10 rows x 7 columns]
client_id          int64
age                int64
job        

In [111]:
print(campaign['previous_outcome'].unique())
print(campaign['campaign_outcome'].unique())

[False  True]
[False  True]


In [112]:
#campaign
# Remove 'last_contact_date' from the column selection since it's not in df
campaign = df[['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'month', 'day']]

#cleaning part
#previous_outcome column
campaign['previous_outcome'] = campaign['previous_outcome'].map({'success': 1, 'nonexistent' : 0, 'failure' : 0}).astype('bool')

#campaign_outcome column
campaign['campaign_outcome'] = campaign['campaign_outcome'].map({'yes': 1, 'no' : 0}).astype('bool')

#last_contact_date column
campaign['last_contact_date'] = '2022' + '-' + campaign['month'] + '-' + campaign['day'].astype(str)
campaign['last_contact_date'] = pd.to_datetime(campaign['last_contact_date'])

#drop month and day
campaign.drop(columns=['month', 'day'], inplace=True)
#checking
print(campaign.head(10))
campaign.head(40)

   client_id  number_contacts  ...  campaign_outcome  last_contact_date
0          0                1  ...             False         2022-05-13
1          1                1  ...             False         2022-05-19
2          2                1  ...             False         2022-05-23
3          3                1  ...             False         2022-05-27
4          4                1  ...             False         2022-05-03
5          5                1  ...             False         2022-05-05
6          6                1  ...             False         2022-05-03
7          7                1  ...             False         2022-05-12
8          8                1  ...             False         2022-05-21
9          9                1  ...             False         2022-05-05

[10 rows x 7 columns]


,client_id,number_contacts,contact_duration,previous_campaign_contacts,previous_outcome,campaign_outcome,last_contact_date
0,0,1,261,0,False,False,2022-05-13
1,1,1,149,0,False,False,2022-05-19
2,2,1,226,0,False,False,2022-05-23
3,3,1,151,0,False,False,2022-05-27
4,4,1,307,0,False,False,2022-05-03
5,5,1,198,0,False,False,2022-05-05
6,6,1,139,0,False,False,2022-05-03
7,7,1,217,0,False,False,2022-05-12
8,8,1,380,0,False,False,2022-05-21
9,9,1,50,0,False,False,2022-05-05


In [113]:
#economics
economics = df[['client_id', 'cons_price_idx', 'euribor_three_months']]

In [114]:
client.to_csv('client.csv', index = False)
campaign.to_csv('campaign.csv', index = False)
economics.to_csv('economics.csv', index = False)